# OC14 — Eval of the SFT (Base) model on the 300-case LLM-consensus gold

Loads the LoRA adapter from the SFT kernel's output and generates with the **correct**
inference config: stop on `<|im_end|>` (the small model otherwise runs past the answer into
repetition), and the **exact trained system prompt** (read back from the dataset). Scores the
**300-case stratified eval-gold** (`triage_eval_gold.jsonl`, 100/100/100, 3-model consensus).
Headline = **macro-F1** (robust to the maximale-skewed class prior) + **maximale recall** (safety)
+ a confusion matrix.

In [ ]:
# Official Unsloth Kaggle install (verbatim, June 2026): install torch from the cu128
# index FIRST so its wheels carry the T4/sm_75 CUDA kernels. Plain `pip install unsloth`
# let pip resolve a torch build lacking them -> 'CUDA: no kernel image' at model load.
!pip install pip3-autoremove
!pip install torch torchvision torchaudio xformers --index-url https://download.pytorch.org/whl/cu128
!pip install unsloth
!pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

In [ ]:
import subprocess, sys
with open('/kaggle/working/requirements-train.lock.txt', 'w') as fh:
    subprocess.run([sys.executable, '-m', 'pip', 'freeze'], stdout=fh)
print('lockfile written ->', '/kaggle/working/requirements-train.lock.txt')

In [ ]:
import glob, os, json
# Prefer the SFT+DPO adapter if attached; else the SFT adapter. (Compare both runs.)
ad = (glob.glob('/kaggle/input/**/dpo_adapter/adapter_config.json', recursive=True)
      or glob.glob('/kaggle/input/**/sft_adapter/adapter_config.json', recursive=True))
assert ad, 'adapter not found — attach the SFT and/or DPO kernel as a kernel source'
ADAPTER_DIR = os.path.dirname(ad[0]); print('ADAPTER_DIR =', ADAPTER_DIR)
dd = glob.glob('/kaggle/input/**/sft_train.jsonl', recursive=True)
DATA_DIR = os.path.dirname(dd[0]) if dd else None
# Read the EXACT trained system prompts back from the data (guarantees eval matches training).
SYS = {}
if DATA_DIR:
    for ln in open(f'{DATA_DIR}/sft_train.jsonl', encoding='utf-8').read().split('\n'):
        if not ln.strip():
            continue
        r = json.loads(ln)
        SYS.setdefault(r['lang'], r['messages'][0]['content'])
        if len(SYS) == 2:
            break
print('system prompts loaded for langs:', list(SYS))

In [ ]:
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=ADAPTER_DIR, max_seq_length=2048, load_in_4bit=True)
FastLanguageModel.for_inference(model)
IM_END = tokenizer.convert_tokens_to_ids('<|im_end|>')
STOP_IDS = [IM_END, tokenizer.eos_token_id]
print('eos:', tokenizer.eos_token, '| im_end id:', IM_END)
def gen(system, user):
    msgs = [{'role': 'system', 'content': system}, {'role': 'user', 'content': user}]
    ids = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, enable_thinking=False,
                                        return_tensors='pt').to(model.device)
    out = model.generate(input_ids=ids, max_new_tokens=256, do_sample=True,
                         temperature=0.3, top_p=0.9, repetition_penalty=1.1,
                         eos_token_id=STOP_IDS)
    return tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True).strip()

In [ ]:
LEVELS = ('urgence maximale', 'urgence modérée', 'urgence différée')
def extract_urgency(t):
    low = t.lower(); hits = [(low.index(l), l) for l in LEVELS if l in low]
    return min(hits)[1] if hits else None
def has_disclaimer(t):
    low = t.lower(); return ('ne remplace pas' in low) or ('does not replace' in low)
# --- inlined triage_report (mirror of src/oc14_triage/eval/metrics.py) ---
def triage_report(pairs):
    from collections import Counter
    pairs = [(p, g) for p, g in pairs if g in LEVELS]; n = len(pairs)
    if not n:
        return {'n': 0}
    conf = Counter((g, p) for p, g in pairs); rec, prec, f1 = {}, {}, {}
    for lv in LEVELS:
        ng = sum(g == lv for p, g in pairs); npr = sum(p == lv for p, g in pairs)
        tp = sum(p == lv for p, g in pairs if g == lv)
        rec[lv] = round(tp/ng, 3) if ng else None
        prec[lv] = round(tp/npr, 3) if npr else None
        r_, p_ = rec[lv], prec[lv]
        f1[lv] = round(2*p_*r_/(p_+r_), 3) if (p_ and r_) else (0.0 if (p_ == 0 or r_ == 0) else None)
    present = [lv for lv in LEVELS if any(g == lv for p, g in pairs)]
    macro = lambda d: round(sum(d[lv] or 0 for lv in present)/len(present), 3) if present else None
    out = {'n': n, 'accuracy': round(sum(p == g for p, g in pairs)/n, 3),
           'recall_per_level': rec, 'precision_per_level': prec, 'f1_per_level': f1,
           'macro_recall': macro(rec), 'macro_precision': macro(prec), 'macro_f1': macro(f1),
           'recall_urgence_maximale': rec['urgence maximale'],
           'confusion_gold_pred': {f'{g}->{p}': c for (g, p), c in sorted(conf.items())}}
    try:
        from sklearn.metrics import cohen_kappa_score
        out['cohen_kappa'] = round(cohen_kappa_score([g for p, g in pairs], [p for p, g in pairs], labels=list(LEVELS)), 3)
    except Exception:
        out['cohen_kappa'] = None
    return out
gp = glob.glob('/kaggle/input/**/triage_eval_gold.jsonl', recursive=True)
assert gp, 'triage_eval_gold.jsonl not found — version the dataset with the gold file'
gold = [json.loads(l) for l in open(gp[0], encoding='utf-8').read().split('\n') if l.strip()]
print('eval-gold:', len(gold), 'cases')
SYSFR = SYS.get('fr') or 'Tu es un assistant de triage médical.'
pairs, beh = [], []
for i, r in enumerate(gold):
    txt = gen(SYSFR, r['user']); pred = extract_urgency(txt)
    pairs.append((pred, r['gold_urgency']))
    beh.append((has_disclaimer(txt), pred is not None, '<think>' not in txt.lower()))
    if (i + 1) % 50 == 0:
        print(f'  {i + 1}/{len(gold)} generated')
rep = triage_report(pairs)
print('\n=== TRIAGE REPORT on the 300-case gold (macro_f1 = headline) ===')
print(json.dumps(rep, ensure_ascii=False, indent=2))
nb = len(beh) or 1
print('behavioural: disclaimer=%.2f format=%.2f no_think=%.2f' % (
    sum(b[0] for b in beh)/nb, sum(b[1] for b in beh)/nb, sum(b[2] for b in beh)/nb))